# Part 2 Embedding generator
In this notebook, you will transform the raw web pages scraped from the HKU Innovation Wing and InnoAcademy into a **vector database** ready for Retrieval-Augmented Generation (RAG).

### What you will learn
- Why we need to **chunk** long documents
- How to use `RecursiveCharacterTextSplitter` effectively
- How to generate high-quality embeddings using Azure OpenAI
- How to store and manage vectors in **ChromaDB**
- Best practices for deterministic IDs and fresh ingestion

By the end of this notebook, you will have a searchable vector database (`Innowing_db`) that a chatbot can query intelligently.

## 2.1. Introduction – From Raw Text to Vector Database

After scraping websites, we have raw HTML-cleaned text.  
However, large language models have limited context windows and cannot efficiently search through thousands of words at once.

**Solution**:  
1. **Split** documents into smaller, meaningful **chunks**  
2. **Convert** each chunk into a dense vector (embedding)  
3. **Store** vectors + metadata in a vector database (ChromaDB)

This process is the foundation of modern **Retrieval-Augmented Generation (RAG)** systems.

## 2.2. Install and Import Libraries

We use the following key packages:
- **`langchain-text-splitters`** → intelligent document chunking
- **`openai`** (Azure) → generate high-quality embeddings
- **`chromadb`** → persistent vector database
- **`python-dotenv`** → securely load API keys

Run the cell below to ensure all dependencies are installed.

In [1]:
%pip install langchain chromadb langchain-text-splitters python-dotenv openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os
import json
import hashlib
from pathlib import Path
from openai import AzureOpenAI
from typing import List, Dict
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from pathlib import Path
BASE_DIR = Path.cwd().parent

c:\Users\Akshay T P\anaconda3\envs\rag-chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2.3. Set Up Your Azure OpenAI API Key

**Security Best Practice**<br>
Never hard-code your API key in the notebook.

**Step-by-step instructions:**
1. Create a new file named **`.env`** in the **same folder** as this notebook (if you haven’t already).
2. Open the `.env` file and add the following line:

```env
AZURE_OPENAI_API_KEY=your_actual_azure_openai_api_key_here
```

3. Save the file. The notebook will automatically load your key using `python-dotenv`.

**Tips:**
1. Make sure the file is named exactly .env (with the dot at the beginning).
2. Do not upload this to GitHub or share it publicly (very important!).
3.  You should have received your personal API key through email. Each student will have 1 million tokens to use. Please do not share your API key with anyone else, or you risk someone else using up all your tokens.

## 2.4. Load Scraped Documents

We load the `data.json` file generated by the web scraper notebook.

**Important**: Always inspect your data before processing!

In [24]:
# Load .env from parent folder
load_dotenv(BASE_DIR / ".env")

DATASET = BASE_DIR / (os.getenv("DATASET") or "data.json")  # Changed to data.json as requested
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL") or "text-embedding-3-small"
CHROMA_PATH = str(BASE_DIR / (os.getenv("CHROMA_PATH") or "chroma_db/chroma_db"))

if os.path.exists(DATASET):
    print(f"📂 Loading documents from {DATASET}...")
    with open(DATASET, "r", encoding="utf-8") as f:
        documents: List[Dict] = json.load(f)
    print(f"   Loaded {len(documents)} raw pages.")
else:
    raise FileNotFoundError(
        f"❌ {DATASET} not found. Please run your scraper first to generate data.json (or set OUTPUT_FILE env var)."
    )

📂 Loading documents from c:\Users\Akshay T P\Desktop\summer-training-chatbot-competition-Akshay-TP\data.json...
   Loaded 20 raw pages.


Check your data before moving to the next step.

In [21]:
# print the first document
print(documents[0])

{'url': 'https://innowings.engg.hku.hk/', 'text': 'Innovation Wing\nSkip to content\nThe Tam Wing Fan Innovation Wing One\nprovides an open environment to foster\ninterdisciplinary innovations among\nundergraduate students\nand teachers in Engineering and Technology. The provision of state-of-the-art facilities in a collaborative space will enable curriculum innovations that emphasize hands-on and experiential learning activities. The Innovation Wing serves as a platform to engage the young generation to explore the world, create opportunities for them to learn about the needs of the underprivileged, and acquire practical hands-on experience in developing solutions with real-world impact.\nInnovation Wing One\nThe Tam Wing Fan Innovation Wing Two\naims to serve as an enabling platform for Engineering\nresearchers\nto interact and collaborate synergistically with researchers and professionals across various disciplines to tackle grand challenges and deliver research outputs with signifi

## 2.5. Text Chunking – Why and How

Long documents must be split into smaller pieces (chunks) because:
- Embeddings perform better on focused, coherent text
- LLMs have token limits
- Smaller chunks improve retrieval precision

We use **`RecursiveCharacterTextSplitter`** — the industry best practice.  
It tries to split on paragraphs → sentences → words while preserving meaning.

**Key parameters**:
- `chunk_size=1000` characters (~250–300 tokens)
- `chunk_overlap=300` characters (prevents loss of context across chunks)

In [20]:
print("\n🔪 Chunking documents using RecursiveCharacterTextSplitter (best-practice settings)...")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    length_function=len,
    separators=[""],  # split only on character count
)

all_texts: List[str] = []
all_metadatas: List[Dict] = []
all_ids: List[str] = []

for doc in documents:
    url = doc["url"]
    text = doc.get("text", "").strip()
    if not text:
        continue

    chunks = text_splitter.split_text(text)

    for chunk_idx, chunk in enumerate(chunks):
        # Deterministic ID (hash of URL + chunk index) → safe re-ingest / upsert
        id_str = f"{url}:{chunk_idx}"
        chunk_id = hashlib.md5(id_str.encode("utf-8")).hexdigest()

        all_texts.append(chunk)
        all_metadatas.append({
            "url": url,
            "source": "HKU InnoWings / InnoAcademy",
            "chunk_index": chunk_idx,
            "total_chunks_on_page": len(chunks),   # useful for debugging / future hierarchical RAG
        })
        all_ids.append(chunk_id)

print(f"Created {len(all_texts)} chunks from {len(documents)} documents.")


🔪 Chunking documents using RecursiveCharacterTextSplitter (best-practice settings)...
Created 62 chunks from 20 documents.


## 2.6.Set Up Azure OpenAI Client

We now load your API key from the `.env` file and create the Azure OpenAI client.

If you see an error about a missing API key, double-check that you have created the `.env` file correctly in the same folder as this notebook.

In [19]:
API_Key = os.getenv("AZURE_OPENAI_API_KEY")
if not API_Key:
    raise RuntimeError("Missing Azure OpenAI credentials.")

client = AzureOpenAI(
    azure_endpoint="https://api-iw.azure-api.net/sig-embedding/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21",
    api_key=API_Key,
    api_version="2024-10-21",
)

def get_embedding(text: str) -> List[float]:
    response = client.embeddings.create(
        input=[text.replace("\n", " ")],
        model=EMBEDDING_MODEL,
    )
    return response.data[0].embedding

## 2.7. ChromaDB Vector Database Setup

**ChromaDB** is a lightweight, developer-friendly, and persistent vector database that is perfect for building local RAG applications.

In this step we will:
- Connect to (or create) a persistent ChromaDB instance on disk
- Delete any existing collection to ensure a **clean, fresh ingestion** (useful during development)
- Create a new collection named `Innowing_db`
- Configure it to use **cosine similarity** as the distance metric — the standard and recommended choice for text embeddings

This setup allows us to efficiently store and later retrieve semantically similar chunks when the chatbot needs to answer questions.

In [17]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

COLLECTION_NAME = "Innowing_db"

try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print(f"🗑️  Deleted existing collection '{COLLECTION_NAME}' for fresh ingest.")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

🗑️  Deleted existing collection 'Innowing_db' for fresh ingest.


## 2.8. Generate Embeddings and Store in ChromaDB

Now comes the core part of the pipeline:

1. We generate a high-quality embedding vector for **every text chunk** using Azure OpenAI’s `text-embedding-3-small` model.
2. We store each embedding together with:
   - The original text (document)
   - Rich metadata (URL, source, chunk index, etc.)
   - A deterministic unique ID (so we can safely re-run this notebook without duplicates)

We process the chunks in **batches of 50** to:
- Respect API rate limits
- Keep memory usage reasonable
- Provide clearer progress feedback

This step may take a few minutes depending on your internet connection and Azure quota.  
Once completed, your vector database will be ready for semantic search and RAG!

In [25]:
print("🧬 Generating embeddings and storing in ChromaDB (this may take a while)...")

BATCH_SIZE = 50

for i in range(0, len(all_texts), BATCH_SIZE):
    batch_texts = all_texts[i : i + BATCH_SIZE]
    batch_metadatas = all_metadatas[i : i + BATCH_SIZE]
    batch_ids = all_ids[i : i + BATCH_SIZE]

    # Generate embeddings for the batch
    embeddings = [get_embedding(text) for text in batch_texts]

    # Use .upsert() instead of .add() → industry standard for idempotent ingestion
    collection.upsert(
        embeddings=embeddings,
        documents=batch_texts,
        ids=batch_ids
    )

    print(f"  [{i + len(batch_texts)}/{len(all_texts)}] Added batch to ChromaDB")

print("\n🎉 Ingestion complete!")
print(f"   ChromaDB collection '{COLLECTION_NAME}' saved to: {CHROMA_PATH}")
print("   ✅ Ready for RAG! Use the same embedding model for queries.")

🧬 Generating embeddings and storing in ChromaDB (this may take a while)...
  [50/62] Added batch to ChromaDB
  [62/62] Added batch to ChromaDB

🎉 Ingestion complete!
   ChromaDB collection 'Innowing_db' saved to: c:\Users\Akshay T P\Desktop\summer-training-chatbot-competition-Akshay-TP\chroma_db\chroma_db
   ✅ Ready for RAG! Use the same embedding model for queries.


## 2.9. Conclusion & Next Steps

**Congratulations!** 🎉  

You have successfully:
- Created a secure `.env` file to protect your API key
- Chunked raw website content into pieces
- Generated embeddings using Azure OpenAI
- Built a persistent vector database with ChromaDB

Your `Innowing_db` is now ready to power a **Retrieval-Augmented Generation (RAG)** chatbot.

### Possible Next Steps
1. Experiment with different chunk sizes or embedding models
2. Try chunking based on other metrics, like indices etc.